<a href="https://colab.research.google.com/github/Hossein-Hesari/Persian-Text-Classification/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning BERT Models for SnappFood Sentiment Analysis

In [ ]:
!pip install -q transformers[torch] datasets evaluate scikit-learn accelerate

## بارگذاری مجموعه داده SnappFood

In [ ]:
from datasets import load_dataset
# بارگذاری دیتاست از هاگینگ‌فیس
dataset = load_dataset('ParsiAI/snappfood-sentiment-analysis')
print(dataset)

# مشکل اصلی: ستون label رشته‌ای است و label_id از نوع float
# Trainer به ستون labels از نوع int نیاز دارد
def prepare_labels(examples):
    # تبدیل label_id (float) به labels (int)
    examples['labels'] = [int(x) for x in examples['label_id']]
    return examples

dataset = dataset.map(prepare_labels, batched=True)
# حذف ستون‌های غیرضروری تا data collator گیج نشود
dataset = dataset.remove_columns(['label', 'label_id'])
print(dataset)
print(dataset['train'][0])
print(dataset['train'].features)


## ۱. فاین‌تیون مدل ParsBERT (HooshvareLab/bert-fa-base-uncased)

In [ ]:
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_bert_name = 'HooshvareLab/bert-fa-base-uncased'
tokenizer_bert = AutoTokenizer.from_pretrained(model_bert_name)

def tokenize_function_bert(examples):
    return tokenizer_bert(examples['comment'], truncation=True, padding='max_length', max_length=128)

# توکنایز کردن کل دیتاست
tokenized_datasets_bert = dataset.map(tokenize_function_bert, batched=True)

# حذف ستون comment چون بعد از توکنایز لازم نیست (و ممکن است collator را گیج کند)
tokenized_datasets_bert = tokenized_datasets_bert.remove_columns(['comment'])

# لود کردن مدل
model_bert = AutoModelForSequenceClassification.from_pretrained(model_bert_name, num_labels=2)

# تعریف متریک‌های ارزیابی
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


In [ ]:
# تنظیمات آموزش برای مدل اول
training_args_bert = TrainingArguments(
    output_dir='./results_bert',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=100,
    report_to='none'
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args_bert,
    train_dataset=tokenized_datasets_bert['train'],
    eval_dataset=tokenized_datasets_bert['validation'],
    compute_metrics=compute_metrics,
)

# اجرای آموزش
trainer_bert.train()

## ۲. فاین‌تیون مدل XLM-RoBERTa

In [ ]:
model_xlmr_name = 'xlm-roberta-base'
tokenizer_xlmr = AutoTokenizer.from_pretrained(model_xlmr_name)

def tokenize_function_xlmr(examples):
    return tokenizer_xlmr(examples['comment'], truncation=True, padding='max_length', max_length=128)

tokenized_datasets_xlmr = dataset.map(tokenize_function_xlmr, batched=True)
tokenized_datasets_xlmr = tokenized_datasets_xlmr.remove_columns(['comment'])

model_xlmr = AutoModelForSequenceClassification.from_pretrained(model_xlmr_name, num_labels=2)

training_args_xlmr = TrainingArguments(
    output_dir='./results_xlmr',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=100,
    report_to='none'
)

trainer_xlmr = Trainer(
    model=model_xlmr,
    args=training_args_xlmr,
    train_dataset=tokenized_datasets_xlmr['train'],
    eval_dataset=tokenized_datasets_xlmr['validation'],
    compute_metrics=compute_metrics,
)

# اجرای آموزش برای مدل دوم
trainer_xlmr.train()
